In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
image_dir = '/content/drive/MyDrive/XAI_Project/results_multi_cam/'
print(f"Updated image directory to: {image_dir}")

Updated image directory to: /content/drive/MyDrive/XAI_Project/results_multi_cam/


In [ ]:
import numpy as np
from scipy.stats import pearsonr
from pathlib import Path
import re  # Potrzebne do wyodrębniania ID obrazu z nazwy pliku
import pandas as pd # Potrzebne do tworzenia DataFrame i zapisu do CSV

# 1. Definiujemy ścieżki do folderów
base_results_dir = Path('/content/drive/MyDrive/XAI_Project/results_multi_cam/')
model_name = 'efficientnet_b0' # Model, dla którego przeprowadzamy analizę
method_a_name = 'EigenCAM'
method_b_name = 'FullGrad'

folder_a = base_results_dir / model_name / method_a_name
folder_b = base_results_dir / model_name / method_b_name

# Lista do przechowywania wyników PCC
pcc_comparison_results = []

print(f"Porównywanie metod {method_a_name} i {method_b_name} dla modelu {model_name}...")

# 2. Iterujemy po plikach map istotności w pierwszym folderze (_raw.npy)
for plik_a_path in folder_a.glob('*_raw.npy'):
    plik_a_name = plik_a_path.name

    # Tworzymy oczekiwaną nazwę pliku w folderze B, zmieniając tylko nazwę metody
    # Przykład: cat_1374_efficientnet_b0_Egyptian_cat_EigenCAM_raw.npy -> cat_1374_efficientnet_b0_Egyptian_cat_FullGrad_raw.npy
    target_name_b = plik_a_name.replace(f'_{method_a_name}_raw.npy', f'_{method_b_name}_raw.npy')
    plik_b_path = folder_b / target_name_b

    # Wyodrębniamy ID obrazu dla lepszego komunikatu
    match_image_id = re.match(r'(cat_\d+|dog_\d+)', plik_a_name) # Dopasowuje np. 'cat_1374' lub 'dog_XYZ'
    image_id = match_image_id.group(0) if match_image_id else plik_a_name.split('_')[0] # Fallback

    if plik_b_path.exists():
        # Wczytujemy dane
        data_a = np.load(plik_a_path)
        data_b = np.load(plik_b_path)

        # PCC wymaga wektorów 1D. Upewniamy się, że oba pliki mają ten sam rozmiar.
        if data_a.shape == data_b.shape:
            # pearsonr zwraca (współczynnik_korelacji, p_value)
            corr, _ = pearsonr(data_a.flatten(), data_b.flatten())

            pcc_comparison_results.append({
                'model': model_name,
                'image_id': image_id,
                'method_a': method_a_name,
                'method_b': method_b_name,
                'PCC': corr
            })
            # print(f"Obraz: {image_id} | Metoda A: {method_a_name}, Metoda B: {method_b_name} | PCC: {corr:.4f}")
        else:
            print(f"⚠️ Błąd: Obrazy {image_id} (z {method_a_name} i {method_b_name}) mają różne wymiary!")
    else:
        # print(f"ℹ️ Brak odpowiednika dla obrazu {image_id} ({method_a_name}) w folderze {method_b_name}. Szukano: {target_name_b}")
        pass # Możemy wyciszyć te komunikaty, jeśli spodziewamy się wielu brakujących par

# Konwersja wyników do DataFrame i zapis do CSV
if pcc_comparison_results:
    df_pcc_comparison = pd.DataFrame(pcc_comparison_results)

    # Nazwa pliku CSV powinna odzwierciedlać model i porównywane metody
    output_filename = f"pcc_{model_name}_{method_a_name}_vs_{method_b_name}_results.csv"
    df_pcc_comparison.to_csv(output_filename, index=False)

    print(f"\nWyniki PCC dla {model_name} ({method_a_name} vs {method_b_name}) zapisano do {output_filename}")
    print("Pierwsze 5 wierszy wyników PCC:")
    display(df_pcc_comparison.head())
elif not pcc_comparison_results and (folder_a.exists() and folder_b.exists()):
    print(f"\nBrak wyników PCC do zapisania. Upewnij się, że pliki _raw.npy istnieją i pasują do wzorca w obu folderach: {folder_a} i {folder_b}")
else:
    print(f"\nBrak wyników PCC do zapisania. Jeden lub oba foldery nie istnieją: {folder_a}, {folder_b}")


Porównywanie metod EigenCAM i FullGrad dla modelu efficientnet_b0...

Wyniki PCC dla efficientnet_b0 (EigenCAM vs FullGrad) zapisano do pcc_efficientnet_b0_EigenCAM_vs_FullGrad_results.csv
Pierwsze 5 wierszy wyników PCC:


,model,image_id,method_a,method_b,PCC
0,efficientnet_b0,cat_1374,EigenCAM,FullGrad,0.694250
1,efficientnet_b0,cat_1375,EigenCAM,FullGrad,0.711488
2,efficientnet_b0,cat_1378,EigenCAM,FullGrad,0.663920
3,efficientnet_b0,cat_1379,EigenCAM,FullGrad,0.728890
4,efficientnet_b0,cat_1382,EigenCAM,FullGrad,0.770329


In [ ]:
from pathlib import Path
import numpy as np
import re

# Define base directory (if not already defined in previous cells)
base_results_dir = Path('/content/drive/MyDrive/XAI_Project/results_multi_cam/')

def load_data_for_metrics_corrected(base_dir):
    all_loaded_data = {}
    # Iterate through model folders (e.g., efficientnet_b0, resnet18)
    for model_folder in base_dir.iterdir():
        if model_folder.is_dir():
            model_name = model_folder.name
            all_loaded_data[model_name] = {}
            # Iterate through method folders within each model (e.g., EigenCAM, FullGrad)
            for method_sub_folder in model_folder.iterdir():
                if method_sub_folder.is_dir():
                    actual_method_name = method_sub_folder.name
                    data_source_path = method_sub_folder

                    # Special handling for LIME's nested structure
                    if actual_method_name.startswith('LIME_Explanations_'):
                        actual_method_name = 'LIME' # Standardize the method name to 'LIME'
                        data_source_path = method_sub_folder / 'Raw_Masks'
                        if not data_source_path.is_dir():
                            print(f"Warning: LIME 'Raw_Masks' folder not found at {data_source_path}. Skipping LIME for {model_name}.")
                            continue # Skip if Raw_Masks not found

                    # Use setdefault to ensure the inner dictionary exists before adding images
                    all_loaded_data[model_name].setdefault(actual_method_name, {})

                    # Iterate through _raw.npy files in the determined data_source_path
                    for saliency_map_path in data_source_path.glob('*_raw.npy'):
                        try:
                            # Extract image ID (e.g., cat_1374) from filename
                            match_image_id = re.match(r'(cat_\d+|dog_\d+)', saliency_map_path.name)
                            image_id = match_image_id.group(0) if match_image_id else saliency_map_path.stem.split('_')[0]

                            # Load the saliency map data
                            saliency_map = np.load(saliency_map_path)
                            all_loaded_data[model_name][actual_method_name][image_id] = saliency_map
                        except Exception as e:
                            print(f"⚠️ Error loading {saliency_map_path}: {e}")
    return all_loaded_data

# Call the function to load all data
print(f"Loading saliency map data from: {base_results_dir}")
all_loaded_data = load_data_for_metrics_corrected(base_results_dir)

# Confirmation message
print("\nSaliency map data loaded successfully!")
print(f"Loaded models: {list(all_loaded_data.keys())}")

# Optional: Display a sample of loaded data keys for verification
if all_loaded_data:
    sample_model = list(all_loaded_data.keys())[0]
    if all_loaded_data[sample_model]:
        sample_method = list(all_loaded_data[sample_model].keys())[0]
        print(f"Sample data structure for model '{sample_model}', method '{sample_method}':")
        print(f"  First 5 image IDs: {list(all_loaded_data[sample_model][sample_method].keys())[:5]}")
    else:
        print(f"No methods found for model '{sample_model}'.")
else:
    print("No data was loaded. Please check the `base_results_dir` path and folder structure.")

Loading saliency map data from: /content/drive/MyDrive/XAI_Project/results_multi_cam

Saliency map data loaded successfully!
Loaded models: ['squeezenet', 'efficientnet_b0', 'efficientnet_b3', 'mobilenet_v3_large', 'resnet18']
Sample data structure for model 'squeezenet', method 'LIME':
  First 5 image IDs: ['cat_1', 'cat_10', 'cat_1001', 'cat_1002', 'cat_1006']


In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from numba import jit
from scipy.stats import pearsonr # For comparison/fallback, though Numba implementation will be used

# 2. Define calculate_pcc function with Numba optimization
@jit(nopython=True)
def calculate_pcc(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # 3. Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error, depending on desired behavior

    # Calculate means
    mean1 = np.mean(flat_arr1)
    mean2 = np.mean(flat_arr2)

    # Calculate numerator (sum of (x_i - mean_x)(y_i - mean_y))
    numerator = np.sum((flat_arr1 - mean1) * (flat_arr2 - mean2))

    # Calculate denominators (sum of (x_i - mean_x)^2 and (y_i - mean_y)^2)
    sum_sq_dev1 = np.sum((flat_arr1 - mean1)**2)
    sum_sq_dev2 = np.sum((flat_arr2 - mean2)**2)

    # Handle potential division by zero (e.g., if one array has zero variance)
    denominator = np.sqrt(sum_sq_dev1) * np.sqrt(sum_sq_dev2)

    if denominator == 0:
        return 0.0 # If there's no variance in one or both, PCC is undefined, return 0
    else:
        pcc = numerator / denominator
        # Handle potential NaN if inputs contain NaNs and lead to NaNs in calculation
        if np.isnan(pcc):
            return 0.0
        return pcc

print("Libraries imported and `calculate_pcc` function defined and optimized with Numba.")

Libraries imported and `calculate_pcc` function defined and optimized with Numba.


In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from numba import jit
from scipy.stats import pearsonr # For comparison/fallback, though Numba implementation will be used

# 2. Define calculate_pcc function with Numba optimization
@jit(nopython=True)
def calculate_pcc(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # 3. Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error, depending on desired behavior

    # Calculate means
    mean1 = np.mean(flat_arr1)
    mean2 = np.mean(flat_arr2)

    # Calculate numerator (sum of (x_i - mean_x)(y_i - mean_y))
    numerator = np.sum((flat_arr1 - mean1) * (flat_arr2 - mean2))

    # Calculate denominators (sum of (x_i - mean_x)^2 and (y_i - mean_y)^2)
    sum_sq_dev1 = np.sum((flat_arr1 - mean1)**2)
    sum_sq_dev2 = np.sum((flat_arr2 - mean2)**2)

    # Handle potential division by zero (e.g., if one array has zero variance)
    denominator = np.sqrt(sum_sq_dev1) * np.sqrt(sum_sq_dev2)

    if denominator == 0:
        return 0.0 # If there's no variance in one or both, PCC is undefined, return 0
    else:
        pcc = numerator / denominator
        # Handle potential NaN if inputs contain NaNs and lead to NaNs in calculation
        if np.isnan(pcc):
            return 0.0
        return pcc

print("Libraries imported and `calculate_pcc` function defined and optimized with Numba.")

start_time = time.time()

pcc_all_models_results = []

# Define the target method for comparison
target_method = 'LIME'

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    comparison_pairs = set() # Use a set to avoid duplicates

    if target_method in method_names:
        # Add pairs (LIME, X) for all X in method_names
        for other_method in method_names:
            comparison_pairs.add((target_method, other_method))
            # Add pairs (X, LIME) for all X in method_names (excluding LIME vs LIME duplicate)
            if other_method != target_method:
                comparison_pairs.add((other_method, target_method))
    else:
        print(f"Warning: The method '{target_method}' is not found in the loaded data for model '{model_name}'. Skipping its comparisons.")
        continue # Skip to the next model if LIME is not found

    # Convert back to list for iteration
    comparison_pairs = list(comparison_pairs)

    # Iterate through the generated pairs
    for method_a_name, method_b_name in comparison_pairs:
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating PCC
            if map_a.shape == map_b.shape:
                pcc_value = calculate_pcc(map_a, map_b)
                pcc_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'PCC': pcc_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping PCC for this pair.")

end_time = time.time()
total_calculation_time = end_time - start_time

print(f"Total PCC calculation time: {total_calculation_time:.2f} seconds")

# Convert results to DataFrame
df_pcc_all_models = pd.DataFrame(pcc_all_models_results)

# Save to CSV
output_csv_filename = '/content/drive/MyDrive/XAI_Project/results_multi_cam/pcc_all_models_lime_focused_results.csv' # Changed filename
df_pcc_all_models.to_csv(output_csv_filename, index=False)

print(f"PCC results focused on LIME comparisons for all models saved to {output_csv_filename}") # Changed message
print("First 5 rows of the results DataFrame:")
display(df_pcc_all_models.head())

Libraries imported and `calculate_pcc` function defined and optimized with Numba.
Total PCC calculation time: 80.66 seconds
PCC full matrix results for all models and method comparisons saved to /content/drive/MyDrive/XAI_Project/results_multi_cam/pcc_all_models_full_matrix_results.csv
First 5 rows of the results DataFrame:


,model,image_id,method_a,method_b,PCC
0,efficientnet_b0,cat_1025,GradCAM,GradCAM,1.0
1,efficientnet_b0,cat_3435,GradCAM,GradCAM,1.0
2,efficientnet_b0,cat_1043,GradCAM,GradCAM,1.0
3,efficientnet_b0,cat_276,GradCAM,GradCAM,1.0
4,efficientnet_b0,cat_2126,GradCAM,GradCAM,1.0


In [ ]:
from pathlib import Path
import numpy as np
import re

# Define base directory (if not already defined in previous cells)
base_results_dir = Path('/content/drive/MyDrive/XAI_Project/results_multi_cam/')

def load_data_for_metrics_corrected(base_dir):
    all_loaded_data = {}
    # Iterate through model folders (e.g., efficientnet_b0, resnet18)
    for model_folder in base_dir.iterdir():
        if model_folder.is_dir():
            model_name = model_folder.name
            all_loaded_data[model_name] = {}
            # Iterate through method folders within each model (e.g., EigenCAM, FullGrad)
            for method_sub_folder in model_folder.iterdir():
                if method_sub_folder.is_dir():
                    actual_method_name = method_sub_folder.name
                    data_source_path = method_sub_folder

                    # Special handling for LIME's nested structure
                    if actual_method_name.startswith('LIME_Explanations_'):
                        actual_method_name = 'LIME' # Standardize the method name to 'LIME'
                        data_source_path = method_sub_folder / 'Raw_Masks'
                        if not data_source_path.is_dir():
                            print(f"Warning: LIME 'Raw_Masks' folder not found at {data_source_path}. Skipping LIME for {model_name}.")
                            continue # Skip if Raw_Masks not found

                    # Use setdefault to ensure the inner dictionary exists before adding images
                    all_loaded_data[model_name].setdefault(actual_method_name, {})

                    # Iterate through _raw.npy files in the determined data_source_path
                    for saliency_map_path in data_source_path.glob('*_raw.npy'):
                        try:
                            # Extract image ID (e.g., cat_1374) from filename
                            match_image_id = re.match(r'(cat_\d+|dog_\d+)', saliency_map_path.name)
                            image_id = match_image_id.group(0) if match_image_id else saliency_map_path.stem.split('_')[0]

                            # Load the saliency map data
                            saliency_map = np.load(saliency_map_path)
                            all_loaded_data[model_name][actual_method_name][image_id] = saliency_map
                        except Exception as e:
                            print(f"⚠️ Error loading {saliency_map_path}: {e}")
    return all_loaded_data

# Call the function to load all data
print(f"Loading saliency map data from: {base_results_dir}")
all_loaded_data = load_data_for_metrics_corrected(base_results_dir)

# Confirmation message
print("\nSaliency map data loaded successfully!")
print(f"Loaded models: {list(all_loaded_data.keys())}")

# Optional: Display a sample of loaded data keys for verification
if all_loaded_data:
    sample_model = list(all_loaded_data.keys())[0]
    if all_loaded_data[sample_model]:
        sample_method = list(all_loaded_data[sample_model].keys())[0]
        print(f"Sample data structure for model '{sample_model}', method '{sample_method}':")
        print(f"  First 5 image IDs: {list(all_loaded_data[sample_model][sample_method].keys())[:5]}")
    else:
        print(f"No methods found for model '{sample_model}'.")
else:
    print("No data was loaded. Please check the `base_results_dir` path and folder structure.")

Loading saliency map data from: /content/drive/MyDrive/XAI_Project/results_multi_cam

Saliency map data loaded successfully!
Loaded models: ['efficientnet_b0', 'efficientnet_b3', 'mobilenet_v3_large', 'resnet18', 'squeezenet']
Sample data structure for model 'efficientnet_b0', method 'GradCAM':
  First 5 image IDs: ['cat_1374', 'cat_1375', 'cat_1378', 'cat_1379', 'cat_1382']


In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from skimage.metrics import structural_similarity as ssim # Import SSIM

# Function to calculate SSIM, expecting 2D arrays
def calculate_ssim(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Ensure both arrays have the same shape
    if arr1.shape != arr2.shape:
        return 0.0 # Or raise an error

    # Normalize arrays to a 0-1 range. This is crucial for SSIM to have a consistent data_range.
    # Saliency maps often have values that aren't strictly 0-1 initially.
    # We normalize each map independently based on its own min/max.
    # Add a small epsilon to avoid division by zero if max-min is zero (flat map)
    epsilon = 1e-8

    if arr1.max() - arr1.min() > epsilon:
        normalized_arr1 = (arr1 - arr1.min()) / (arr1.max() - arr1.min())
    else:
        normalized_arr1 = np.zeros_like(arr1) # If map is flat, normalize to all zeros

    if arr2.max() - arr2.min() > epsilon:
        normalized_arr2 = (arr2 - arr2.min()) / (arr2.max() - arr2.min())
    else:
        normalized_arr2 = np.zeros_like(arr2) # If map is flat, normalize to all zeros

    # Calculate SSIM. data_range=1.0 because we normalized to 0-1.
    # If the maps are 3D (e.g., RGB), channel_axis might be needed.
    # Assuming saliency maps are grayscale (2D) or will be treated as such.
    ssim_value = ssim(normalized_arr1, normalized_arr2, data_range=1.0)
    return ssim_value

print("SSIM function defined.")

start_time_ssim = time.time()

ssim_all_models_results = []

# Define the target method for comparison
target_method = 'LIME'

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    comparison_pairs = set() # Use a set to avoid duplicates

    if target_method in method_names:
        # Add pairs (LIME, X) for all X in method_names
        for other_method in method_names:
            comparison_pairs.add((target_method, other_method))
            # Add pairs (X, LIME) for all X in method_names (excluding LIME vs LIME duplicate)
            if other_method != target_method:
                comparison_pairs.add((other_method, target_method))
    else:
        print(f"Warning: The method '{target_method}' is not found in the loaded data for model '{model_name}'. Skipping its comparisons.")
        continue # Skip to the next model if LIME is not found

    # Convert back to list for iteration
    comparison_pairs = list(comparison_pairs)

    # Iterate through the generated pairs
    for method_a_name, method_b_name in comparison_pairs:
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating SSIM
            if map_a.shape == map_b.shape:
                ssim_value = calculate_ssim(map_a, map_b)
                ssim_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'SSIM': ssim_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping SSIM for this pair.")

end_time_ssim = time.time()
total_calculation_time_ssim = end_time_ssim - start_time_ssim

print(f"Total SSIM calculation time: {total_calculation_time_ssim:.2f} seconds")

# Convert results to DataFrame
df_ssim_all_models = pd.DataFrame(ssim_all_models_results)

# Save to CSV in the same directory as PCC results
output_csv_filename_ssim = '/content/drive/MyDrive/XAI_Project/results_multi_cam/ssim_all_models_lime_focused_results.csv' # Changed filename
df_ssim_all_models.to_csv(output_csv_filename_ssim, index=False)

print(f"SSIM results focused on LIME comparisons for all models saved to {output_csv_filename_ssim}") # Changed message
print("First 5 rows of the SSIM results DataFrame:")
display(df_ssim_all_models.head())

SSIM function defined.
Total SSIM calculation time: 498.98 seconds
SSIM full matrix results for all models and method comparisons saved to /content/drive/MyDrive/XAI_Project/results_multi_cam/ssim_all_models_full_matrix_results.csv
First 5 rows of the SSIM results DataFrame:


,model,image_id,method_a,method_b,SSIM
0,efficientnet_b0,cat_1025,GradCAM,GradCAM,1.0
1,efficientnet_b0,cat_3435,GradCAM,GradCAM,1.0
2,efficientnet_b0,cat_1043,GradCAM,GradCAM,1.0
3,efficientnet_b0,cat_276,GradCAM,GradCAM,1.0
4,efficientnet_b0,cat_2126,GradCAM,GradCAM,1.0


In [ ]:
import time
import itertools
import pandas as pd
import numpy as np
from scipy.stats import spearmanr # Import Spearman's correlation

# Function to calculate Spearman's Rank Correlation Coefficient
def calculate_spearmanr(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error

    # spearmanr returns (correlation_coefficient, p_value)
    corr, _ = spearmanr(flat_arr1, flat_arr2)

    # Handle potential NaN values (e.g., if one array has zero variance or all values are the same)
    if np.isnan(corr):
        return 0.0
    return corr

print("Spearman's Rank Correlation function defined.")

Spearman's Rank Correlation function defined.


In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from scipy.stats import spearmanr # Import Spearman's correlation

# Function to calculate Spearman's Rank Correlation Coefficient
def calculate_spearmanr(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error

    # spearmanr returns (correlation_coefficient, p_value)
    corr, _ = spearmanr(flat_arr1, flat_arr2)

    # Handle potential NaN values (e.g., if one array has zero variance or all values are the same)
    if np.isnan(corr):
        return 0.0
    return corr

print("Spearman's Rank Correlation function defined.")

start_time_spearmanr = time.time()

spearmanr_all_models_results = []

# Define the target method for comparison
target_method = 'LIME'

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    comparison_pairs = set() # Use a set to avoid duplicates

    if target_method in method_names:
        # Add pairs (LIME, X) for all X in method_names
        for other_method in method_names:
            comparison_pairs.add((target_method, other_method))
            # Add pairs (X, LIME) for all X in method_names (excluding LIME vs LIME duplicate)
            if other_method != target_method:
                comparison_pairs.add((other_method, target_method))
    else:
        print(f"Warning: The method '{target_method}' is not found in the loaded data for model '{model_name}'. Skipping its comparisons.")
        continue # Skip to the next model if LIME is not found

    # Convert back to list for iteration
    comparison_pairs = list(comparison_pairs)

    # Iterate through the generated pairs
    for method_a_name, method_b_name in comparison_pairs:
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating Spearman's
            if map_a.shape == map_b.shape:
                spearmanr_value = calculate_spearmanr(map_a, map_b)
                spearmanr_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'SpearmanR': spearmanr_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping SpearmanR for this pair.")

end_time_spearmanr = time.time()
total_calculation_time_spearmanr = end_time_spearmanr - start_time_spearmanr

print(f"Total Spearman's Rank Correlation calculation time: {total_calculation_time_spearmanr:.2f} seconds")

# Convert results to DataFrame
df_spearmanr_all_models = pd.DataFrame(spearmanr_all_models_results)

# Save to CSV in the same directory as PCC results
output_csv_filename_spearmanr = '/content/drive/MyDrive/XAI_Project/results_multi_cam/spearmanr_all_models_lime_focused_results.csv' # Changed filename
df_spearmanr_all_models.to_csv(output_csv_filename_spearmanr, index=False)

print(f"Spearman's Rank Correlation results focused on LIME comparisons for all models saved to {output_csv_filename_spearmanr}") # Changed message
print("First 5 rows of the Spearman's R results DataFrame:")
display(df_spearmanr_all_models.head())

Spearman's Rank Correlation function defined.


/tmp/ipython-input-2197119555.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = spearmanr(flat_arr1, flat_arr2)


Total Spearman's Rank Correlation calculation time: 1736.51 seconds
Spearman's Rank Correlation full matrix results for all models and method comparisons saved to /content/drive/MyDrive/XAI_Project/results_multi_cam/spearmanr_all_models_full_matrix_results.csv
First 5 rows of the Spearman's R results DataFrame:


,model,image_id,method_a,method_b,SpearmanR
0,efficientnet_b0,cat_1025,GradCAM,GradCAM,1.0
1,efficientnet_b0,cat_3435,GradCAM,GradCAM,1.0
2,efficientnet_b0,cat_1043,GradCAM,GradCAM,1.0
3,efficientnet_b0,cat_276,GradCAM,GradCAM,1.0
4,efficientnet_b0,cat_2126,GradCAM,GradCAM,1.0


In [ ]:
import time
import itertools
import pandas as pd
import numpy as np
from sklearn.metrics import matthews_corrcoef # Import MCC
from scipy.special import rel_entr # For KL Divergence

# Function to calculate Matthews Correlation Coefficient
def calculate_mcc(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error

    # Binarize maps using their respective medians as thresholds
    # Handle constant arrays: if all values are the same, median is that value.
    # Using > median ensures there's at least some '1' if not all values are identical.

    if np.all(flat_arr1 == flat_arr1[0]): # Check if arr1 is constant
        bin_arr1 = np.zeros_like(flat_arr1, dtype=bool)
    else:
        median_arr1 = np.median(flat_arr1)
        bin_arr1 = flat_arr1 > median_arr1

    if np.all(flat_arr2 == flat_arr2[0]): # Check if arr2 is constant
        bin_arr2 = np.zeros_like(flat_arr2, dtype=bool)
    else:
        median_arr2 = np.median(flat_arr2)
        bin_arr2 = flat_arr2 > median_arr2

    # matthews_corrcoef raises ValueError if input contains only one class
    # or if the confusion matrix leads to division by zero (e.g., all True or all False)
    if len(np.unique(bin_arr1)) < 2 or len(np.unique(bin_arr2)) < 2:
        return 0.0 # MCC is undefined in such cases, return 0.0 as a neutral value

    try:
        mcc = matthews_corrcoef(bin_arr1, bin_arr2)
        return mcc
    except ValueError: # Catch cases where MCC cannot be computed (e.g., all predictions are the same)
        return 0.0 # Return 0.0 for undefined MCC

# Function to calculate KL Divergence
def calculate_kl_divergence(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    p = arr1.flatten()
    q = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(p) != len(q):
        return np.nan # Or raise an error

    # Normalize to form probability distributions
    # Add a small epsilon to avoid division by zero when normalizing or taking log
    epsilon = 1e-10
    p = p + epsilon
    q = q + epsilon

    p_sum = np.sum(p)
    q_sum = np.sum(q)

    if p_sum == 0 or q_sum == 0:
        return np.nan # Cannot calculate KL if one distribution sums to zero

    p_normalized = p / p_sum
    q_normalized = q / q_sum

    # Calculate KL Divergence using scipy.special.rel_entr
    # rel_entr(p, q) calculates p * log(p/q) element-wise
    kl_div = np.sum(rel_entr(p_normalized, q_normalized))
    return kl_div

print("Matthews Correlation Coefficient (MCC) and Kullback-Leibler (KL) Divergence functions defined.")

Matthews Correlation Coefficient (MCC) and Kullback-Leibler (KL) Divergence functions defined.


In [ ]:
import time
import itertools
import pandas as pd
import numpy as np
from sklearn.metrics import matthews_corrcoef # Import MCC
from scipy.special import rel_entr # For KL Divergence

# Function to calculate Matthews Correlation Coefficient
def calculate_mcc(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error

    # Binarize maps using their respective medians as thresholds
    # Handle constant arrays: if all values are the same, median is that value.
    # Using > median ensures there's at least some '1' if not all values are identical.

    if np.all(flat_arr1 == flat_arr1[0]): # Check if arr1 is constant
        bin_arr1 = np.zeros_like(flat_arr1, dtype=bool)
    else:
        median_arr1 = np.median(flat_arr1)
        bin_arr1 = flat_arr1 > median_arr1

    if np.all(flat_arr2 == flat_arr2[0]): # Check if arr2 is constant
        bin_arr2 = np.zeros_like(flat_arr2, dtype=bool)
    else:
        median_arr2 = np.median(flat_arr2)
        bin_arr2 = flat_arr2 > median_arr2

    # matthews_corrcoef raises ValueError if input contains only one class
    # or if the confusion matrix leads to division by zero (e.g., all True or all False)
    if len(np.unique(bin_arr1)) < 2 or len(np.unique(bin_arr2)) < 2:
        return 0.0 # MCC is undefined in such cases, return 0.0 as a neutral value

    try:
        mcc = matthews_corrcoef(bin_arr1, bin_arr2)
        return mcc
    except ValueError: # Catch cases where MCC cannot be computed (e.g., all predictions are the same)
        return 0.0 # Return 0.0 for undefined MCC

# Function to calculate KL Divergence
def calculate_kl_divergence(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    p = arr1.flatten()
    q = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(p) != len(q):
        return np.nan # Or raise an error

    # Normalize to form probability distributions
    # Add a small epsilon to avoid division by zero when normalizing or taking log
    epsilon = 1e-10
    p = p + epsilon
    q = q + epsilon

    p_sum = np.sum(p)
    q_sum = np.sum(q)

    if p_sum == 0 or q_sum == 0:
        return np.nan # Cannot calculate KL if one distribution sums to zero

    p_normalized = p / p_sum
    q_normalized = q / q_sum

    # Calculate KL Divergence using scipy.special.rel_entr
    # rel_entr(p, q) calculates p * log(p/q) element-wise
    kl_div = np.sum(rel_entr(p_normalized, q_normalized))
    return kl_div

start_time_mcc = time.time()

mcc_all_models_results = []
kl_div_all_models_results = []

# Define the target method for comparison
target_method = 'LIME'

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    comparison_pairs = set() # Use a set to avoid duplicates

    if target_method in method_names:
        # Add pairs (LIME, X) for all X in method_names
        for other_method in method_names:
            comparison_pairs.add((target_method, other_method))
            # Add pairs (X, LIME) for all X in method_names (excluding LIME vs LIME duplicate)
            if other_method != target_method:
                comparison_pairs.add((other_method, target_method))
    else:
        print(f"Warning: The method '{target_method}' is not found in the loaded data for model '{model_name}'. Skipping its comparisons.")
        continue # Skip to the next model if LIME is not found

    # Convert back to list for iteration
    comparison_pairs = list(comparison_pairs)

    # Iterate through the generated pairs
    for method_a_name, method_b_name in comparison_pairs:
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating metrics
            if map_a.shape == map_b.shape:
                # Calculate MCC
                mcc_value = calculate_mcc(map_a, map_b)
                mcc_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'MCC': mcc_value
                })

                # Calculate KL Divergence
                kl_div_value = calculate_kl_divergence(map_a, map_b)
                kl_div_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'KL_Divergence': kl_div_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping MCC and KL for this pair.")

end_time_mcc = time.time()
total_calculation_time_mcc = end_time_mcc - start_time_mcc

print(f"Total MCC and KL Divergence calculation time: {total_calculation_time_mcc:.2f} seconds")

# Convert MCC results to DataFrame and save to CSV
df_mcc_all_models = pd.DataFrame(mcc_all_models_results)
output_csv_filename_mcc = '/content/drive/MyDrive/XAI_Project/results_multi_cam/mcc_all_models_lime_focused_results.csv' # Changed filename
df_mcc_all_models.to_csv(output_csv_filename_mcc, index=False)
print(f"MCC results focused on LIME comparisons for all models saved to {output_csv_filename_mcc}") # Changed message
print("First 5 rows of the MCC results DataFrame:")
display(df_mcc_all_models.head())

# Convert KL Divergence results to DataFrame and save to CSV
df_kl_div_all_models = pd.DataFrame(kl_div_all_models_results)
output_csv_filename_kl = '/content/drive/MyDrive/XAI_Project/results_multi_cam/kl_divergence_all_models_lime_focused_results.csv' # Changed filename
df_kl_div_all_models.to_csv(output_csv_filename_kl, index=False)
print(f"KL Divergence results focused on LIME comparisons for all models saved to {output_csv_filename_kl}") # Changed message
print("First 5 rows of the KL Divergence results DataFrame:")
display(df_kl_div_all_models.head())